# OSM Roads for ArcGIS Velocity and GeoAnalytics Engine

Prepares OpenStreetMap road networks for:
- **ArcGIS Velocity**: [Snap to Network](https://doc.arcgis.com/en/velocity/analyze/snap-to-network.htm)
- **ArcGIS GeoAnalytics Engine**: [Snap Tracks](https://developers.arcgis.com/geoanalytics-fabric/tools/snap-tracks/)

Extracts drivable roads from Geofabrik PBF files and generates required schema fields for both tools.

## Step 1: Configuration

In [3]:
import os

# Bounding box (west, south, east, north) in WGS84
# BBOX = (153.2, -28.3, 153.7, -27.8)  # Gold Coast
BBOX = (152.6, -27.7, 153.3, -27.2)  # Brisbane metro
# BBOX = (152.4, -28.3, 153.7, -26.5)  # SEQ region

# Input PBF
INPUT_PBF = r"C:\Work\Brisbane_GTFS\Incoming\australia-260330.osm.pbf"

# Working directory
WORK_DIR = r"C:\Work\Brisbane_GTFS\SEQ_Network"

# Output
OUTPUT_NAME = "brisbane_roads"
OUTPUT_GDB = os.path.join(WORK_DIR, f"{OUTPUT_NAME.title().replace('_', '')}.gdb")
FC_NAME = f"{OUTPUT_NAME.title().replace('_', '')}Network"

# Highway types to include (drivable roads)
HIGHWAY_FILTER = [
    'motorway', 'trunk', 'primary', 'secondary', 'tertiary',
    'unclassified', 'residential', 'service',
    'motorway_link', 'trunk_link', 'primary_link',
    'secondary_link', 'tertiary_link'
]

EXCLUDE_DRIVEWAYS = True

print(f"Bbox: {BBOX}")
print(f"PBF: {INPUT_PBF}")
print(f"Output: {OUTPUT_GDB}/{FC_NAME}")

Bbox: (152.6, -27.7, 153.3, -27.2)
PBF: C:\Work\Brisbane_GTFS\Incoming\australia-260330.osm.pbf
Output: C:\Work\Brisbane_GTFS\SEQ_Network\BrisbaneRoads.gdb/BrisbaneRoadsNetwork


## Step 2: Environment check

In [4]:
import sys
import arcpy

print(f"Python: {sys.version.split()[0]}")
print(f"ArcPy: {arcpy.GetInstallInfo()['Version']}")

for lib in ['pyogrio', 'geopandas', 'pandas']:
    try:
        mod = __import__(lib)
        print(f"✓ {lib} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"✗ {lib} MISSING")
        raise

if not os.path.exists(INPUT_PBF):
    raise FileNotFoundError(f"PBF not found: {INPUT_PBF}")

size_mb = os.path.getsize(INPUT_PBF) / 1024 / 1024
print(f"\n✓ Input PBF: {size_mb:.0f} MB")

Python: 3.13.7
ArcPy: 3.6.3
✓ pyogrio 0.12.1
✓ geopandas 1.1.0
✓ pandas 2.3.0

✓ Input PBF: 886 MB


## Step 3: Extract roads from PBF

In [5]:
import pyogrio
import geopandas as gpd
import pandas as pd

print("Extracting roads from OSM PBF...")
print(f"  Bbox: {BBOX}")

gdf = pyogrio.read_dataframe(
    INPUT_PBF,
    layer='lines',
    bbox=BBOX,
    where=f"highway IN {tuple(HIGHWAY_FILTER)}",
    use_arrow=True
)

print(f"  Initial: {len(gdf):,} features")

# Remove driveways
if EXCLUDE_DRIVEWAYS and 'other_tags' in gdf.columns:
    before = len(gdf)
    gdf = gdf[~gdf['other_tags'].str.contains('"service"=>"driveway"', na=False)]
    if before > len(gdf):
        print(f"  Removed {before - len(gdf):,} driveways")

# Explode MultiLineStrings
if (gdf.geom_type == 'MultiLineString').any():
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)

print(f"\n✓ Extracted: {len(gdf):,} features")
print("\nTop highway types:")
for hwy, count in gdf['highway'].value_counts().head(5).items():
    print(f"  {hwy:20s} {count:6,} ({100*count/len(gdf):4.1f}%)")

Extracting roads from OSM PBF...
  Bbox: (152.6, -27.7, 153.3, -27.2)
  Initial: 141,228 features
  Removed 20,998 driveways

✓ Extracted: 120,230 features

Top highway types:
  residential          44,690 (37.2%)
  service              28,409 (23.6%)
  tertiary             11,993 (10.0%)
  secondary            10,508 ( 8.7%)
  primary               8,176 ( 6.8%)


## Step 4: Generate Velocity direction fields

In [6]:
print("Generating Velocity direction fields...")

gdf['F_AUTOMOBILES'] = 'N'
gdf['T_AUTOMOBILES'] = 'N'

# Handle oneway tag
if 'oneway' in gdf.columns:
    mask_fwd = gdf['oneway'] == 'yes'
    gdf.loc[mask_fwd, 'T_AUTOMOBILES'] = 'Y'
    print(f"  One-way (forward): {mask_fwd.sum():,}")
    
    mask_rev = gdf['oneway'] == '-1'
    gdf.loc[mask_rev, 'F_AUTOMOBILES'] = 'Y'
    gdf.loc[mask_rev, 'T_AUTOMOBILES'] = 'N'
    if mask_rev.sum() > 0:
        print(f"  One-way (reverse): {mask_rev.sum():,}")

print("\n✓ Complete")

Generating Velocity direction fields...

✓ Complete


## Step 5: Generate GeoAnalytics connectivity

In [7]:
print("Generating GeoAnalytics connectivity...")

# Extract unique nodes from endpoints
nodes = set()
for geom in gdf.geometry:
    coords = list(geom.coords)
    nodes.add(coords[0])
    nodes.add(coords[-1])

# Create node ID mapping
node_to_id = {coord: i for i, coord in enumerate(sorted(nodes))}

# Assign from_node and to_node
def get_nodes(geom):
    coords = list(geom.coords)
    return pd.Series([node_to_id[coords[0]], node_to_id[coords[-1]]])

gdf[['from_node', 'to_node']] = gdf.geometry.apply(get_nodes)

# Direction field
def map_dir(r):
    f, t = r['F_AUTOMOBILES'], r['T_AUTOMOBILES']
    if f=='N' and t=='Y': return 'FT'
    elif f=='Y' and t=='N': return 'TF'
    elif f=='N' and t=='N': return 'B'
    return 'N'

gdf['direction'] = gdf.apply(map_dir, axis=1)

print(f"  Unique nodes: {len(nodes):,}")
print(f"  Node range: {gdf['from_node'].min()} to {gdf['to_node'].max()}")
print("✓ Complete")

Generating GeoAnalytics connectivity...
  Unique nodes: 148,065
  Node range: 0 to 148062
✓ Complete


## Step 6: Prepare output schema

In [8]:
# Select output columns
cols = ['from_node', 'to_node', 'direction', 'F_AUTOMOBILES', 'T_AUTOMOBILES', 'highway', 'name', 'osm_id', 'geometry']
avail = [c for c in cols if c in gdf.columns or c=='geometry']
gdf_output = gdf[avail].copy()

# Fill nulls in name field (proper way to avoid pandas warning)
if 'name' in gdf_output.columns:
    gdf_output = gdf_output.assign(name=gdf_output['name'].fillna(''))

print(f"✓ {len(gdf_output):,} features ready")

✓ 120,230 features ready


## Step 7: Write to File Geodatabase

In [9]:
import tempfile

arcpy.env.overwriteOutput = True

# Create GDB if needed
if not arcpy.Exists(OUTPUT_GDB):
    arcpy.management.CreateFileGDB(os.path.dirname(OUTPUT_GDB), os.path.basename(OUTPUT_GDB))
    print(f"Created GDB: {OUTPUT_GDB}")

# Write via temporary GeoPackage (GeoDataFrame -> GDB requires intermediate format)
with tempfile.TemporaryDirectory() as tmpdir:
    tmp_gpkg = os.path.join(tmpdir, 'temp.gpkg')
    gdf_output.to_file(tmp_gpkg, driver='GPKG', layer='network')
    
    output_fc = os.path.join(OUTPUT_GDB, FC_NAME)
    arcpy.conversion.FeatureClassToFeatureClass(f"{tmp_gpkg}\\network", OUTPUT_GDB, FC_NAME)

print(f"✓ Written: {int(arcpy.management.GetCount(output_fc)[0]):,} features")

Created GDB: C:\Work\Brisbane_GTFS\SEQ_Network\BrisbaneRoads.gdb
✓ Written: 120,230 features


## Step 8: Validation

In [10]:
fc = output_fc
flds = [f.name for f in arcpy.ListFields(fc)]

print("\nVelocity requirements:")
for f in ['OBJECTID', 'F_AUTOMOBILES', 'T_AUTOMOBILES']:
    print(f"  {f:20s} {'✓' if f in flds else '✗'}")

print("\nGeoAnalytics requirements:")
for f in ['from_node', 'to_node', 'direction']:
    print(f"  {f:20s} {'✓' if f in flds else '✗'}")

print(f"\n✓ COMPLETE - {int(arcpy.management.GetCount(fc)[0]):,} features")


Velocity requirements:
  OBJECTID             ✓
  F_AUTOMOBILES        ✓
  T_AUTOMOBILES        ✓

GeoAnalytics requirements:
  from_node            ✓
  to_node              ✓
  direction            ✓

✓ COMPLETE - 120,230 features


## Publishing

1. Add to map → Share As Web Layer → Feature (hosted)
2. Enable Feature Access, disable Editing/Sync
3. Use in Velocity Snap to Network or GeoAnalytics Snap Tracks